In [1]:
import datasets
import pandas as pd
import json
from pathlib import Path
from collections import Counter

In [2]:
!pip install -q datasets

In [16]:
import json
import random
from pathlib import Path
from collections import defaultdict, Counter
from datasets import load_dataset


In [5]:
d2_answerable    = []
d2_missing_ctx   = []

with open("C:\\Users\\arjun\\OneDrive\\Documents\\Stanford\\Coterm\\Winter\\CS525\\abstention-data\\data\\dataset2.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["output"] == NO_ANSWER_TOKEN:
            d2_missing_ctx.append(ex)
        else:
            d2_answerable.append(ex)

print(f"Dataset 2 — answerable:      {len(d2_answerable)}")
print(f"Dataset 2 — missing context: {len(d2_missing_ctx)}")

Dataset 2 — answerable:      5000
Dataset 2 — missing context: 5000


In [23]:
import json
from pathlib import Path

abstention_bench_path = Path("data/abstentionbench/abstention_bench_dataset.jsonl")

abstention_bench_train = []

with abstention_bench_path.open("r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        # Keep only examples NOT used in dataset4
        if not ex.get("used_in_dataset4", False):
            abstention_bench_train.append(ex)

print(f"Total examples: {len(abstention_bench_train):,}")

# Just the questions (or inputs) for unused examples
unused_questions = [
    ex.get("question", ex.get("input", "")).strip()
    for ex in abstention_bench_train
]

# Peek at a few
for q in abstention_bench_train[:5]:
    print("---")
    print(q)

Total examples: 14,179
---
{'question': 'You are a powerful question-answering system with knowledge in the field of biology.\nUsers will provide some biological information along with a question.\nYour task is to combine the information provided by the user with your biological knowledge to answer the question.\nIf you are unable to answer the question, simply respond with "I don\'t know."\nHere is the basic information about a taxon you can refer:\n###\n{\'name\': \'inyidiidae\',\n \'property\': {\'Body symmetry\': [\'sinistrally coiled\'],\n              \'cellularity\': [\'unicellular\'],\n              \'geographic distribution\': [\'Mozambique\'],\n              \'latitude\': [\'10.0 degrees\', \'50.7729 degrees\'],\n              \'longitude\': [\'-11.8022 degrees\'],\n              \'prey on\': [\'Thripidae\', \'Cecidomyiidae\'],\n              \'records in bold?\': [\'yes\'],\n              \'records in gbif?\': [\'yes\'],\n              \'references in bhl?\': [\'yes\'],\n   

In [25]:
from collections import Counter

scenarios = [ex.get("scenario", "UNKNOWN") for ex in abstention_bench_train]
scenario_counts = Counter(scenarios)

print("\nScenario distribution among unused examples:")
for scenario, count in scenario_counts.most_common():
    print(f"  {scenario}: {count:,}")


Scenario distribution among unused examples:
  Underspecified Context: 6,877
  Unknown: 4,367
  Answer Unknown: 1,469
  False Premise: 1,036
  Subjective: 430


In [28]:
import json
import random
from pathlib import Path
from collections import defaultdict, Counter

random.seed(42)

# --- 1) Group AbstentionBench (unused) by scenario ---
scenario_groups = defaultdict(list)
for ex in abstention_bench_train:
    scenario = ex.get("scenario", "Unknown")
    scenario_groups[scenario].append(ex)

for scenario, group in scenario_groups.items():
    print(f"{scenario}: {len(group):,} examples")

# --- 2) Select unanswerable examples per your plan (no synthetic yet) ---

TARGET_PER_SCENARIO = 1250

def take_examples(scenario, target, allow_all_if_insufficient=False):
    pool = scenario_groups.get(scenario, [])
    if not pool:
        print(f"[WARN] No examples for scenario: {scenario}")
        return []

    if allow_all_if_insufficient and len(pool) < target:
        print(f"{scenario}: taking ALL {len(pool)} (less than target {target})")
        return pool.copy()

    n = min(target, len(pool))
    print(f"{scenario}: sampling {n} of {len(pool)}")
    return random.sample(pool, n)

selected_unanswerable = []

# Enough examples: sample 1250
selected_unanswerable += take_examples("Answer Unknown", TARGET_PER_SCENARIO, allow_all_if_insufficient=False)
selected_unanswerable += take_examples("Underspecified Context", TARGET_PER_SCENARIO, allow_all_if_insufficient=False)

# Not enough: take all (will fill the rest synthetically later)
selected_unanswerable += take_examples("Subjective", TARGET_PER_SCENARIO, allow_all_if_insufficient=True)
selected_unanswerable += take_examples("False Premise", TARGET_PER_SCENARIO, allow_all_if_insufficient=True)

print(f"\nTotal unanswerable selected (real, no synthetic): {len(selected_unanswerable):,}")

scenario_counts = Counter(ex.get("scenario", "Unknown") for ex in selected_unanswerable)
print("\nUnanswerable scenario counts in dataset3 (before synthetic):")
for scenario, count in sorted(scenario_counts.items()):
    print(f"  {scenario}: {count:,}")

# --- 3) Convert selected AbstentionBench examples into dataset3 format ---

d3_unanswerable = []
for i, ex in enumerate(selected_unanswerable):
    d3_unanswerable.append({
        "id": ex.get("id", f"abstention_bench_{i}"),  # synthetic id if none present
        "input": ex.get("question", ex.get("input", "")).strip(),
        "output": "<NO-ANSWER>",
        "split": "dev",
        # Keep scenario so we know unanswerable type for later analysis/synthetic filling
        "unanswerable_type": ex.get("scenario", "Unknown"),
    })

print(f"\nConverted to dataset3-format unanswerable: {len(d3_unanswerable):,}")

# --- 4) Combine with the 5,000 answerable examples from dataset2 ---

# d2_answerable already loaded earlier:
#   len(d2_answerable) == 5000

dataset3 = d2_answerable + d3_unanswerable
random.shuffle(dataset3)

print(f"\nFinal dataset3 size (current, no synthetic): {len(dataset3):,}")
print(f"  Answerable:   {len(d2_answerable):,}")
print(f"  Unanswerable: {len(d3_unanswerable):,}")

# --- 5) Save to disk ---

output_path = Path("data/dataset3.jsonl")
with output_path.open("w", encoding="utf-8") as f:
    for ex in dataset3:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"\n✓ Saved dataset3 to {output_path}")

Underspecified Context: 6,877 examples
Unknown: 4,367 examples
Answer Unknown: 1,469 examples
False Premise: 1,036 examples
Subjective: 430 examples
Answer Unknown: sampling 1250 of 1469
Underspecified Context: sampling 1250 of 6877
Subjective: taking ALL 430 (less than target 1250)
False Premise: taking ALL 1036 (less than target 1250)

Total unanswerable selected (real, no synthetic): 3,966

Unanswerable scenario counts in dataset3 (before synthetic):
  Answer Unknown: 1,250
  False Premise: 1,036
  Subjective: 430
  Underspecified Context: 1,250

Converted to dataset3-format unanswerable: 3,966

Final dataset3 size (current, no synthetic): 8,966
  Answerable:   5,000
  Unanswerable: 3,966

✓ Saved dataset3 to data\dataset3.jsonl


In [35]:
import json
from pathlib import Path

DATA_DIR = Path("data")
d1_path = DATA_DIR / "dataset1.jsonl"
d2_path = DATA_DIR / "dataset2.jsonl"

# IDs used anywhere in dataset2
used_in_d2 = set()
with d2_path.open("r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        used_in_d2.add(ex["id"])

print(f"IDs used in dataset2: {len(used_in_d2):,}")

# Seed pool = dataset1 minus any IDs that appear in dataset2
seed_squad = []
with d1_path.open("r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in used_in_d2:
            continue
        seed_squad.append({
            "id": ex["id"],
            "question": ex["input"],
            "answer": ex["output"],  # original SQuAD answer
            "split": ex["split"],
        })

print(f"SQuAD-based seed pool (dataset1 minus dataset2): {len(seed_squad):,}")
print("Example seed:")
print(seed_squad[0])

IDs used in dataset2: 10,000
SQuAD-based seed pool (dataset1 minus dataset2): 5,000
Example seed:
{'id': '571a30bb10f8ca1400304f53', 'question': 'To what two islands does the ferry service connect?', 'answer': 'Bainbridge and Vashon', 'split': 'train'}


In [ ]:
import os
import json
import random
from typing import List, Dict
from openai import OpenAI

# ---- OpenAI client ----
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# How many synthetic False Premise questions you still need
TARGET_FALSE_PREMISE = 214  # 1250 target - 1036 real from AbstentionBench

# ---- Prompt builder ----

def make_false_premise_prompt(question: str, answer: str | None = None) -> str:
    extra = " and its answer" if answer else ""
    answer_block = f'\nOriginal factual answer: "{answer}"\n' if answer else ""
    return f"""
You are helping build an abstention benchmark.

Definition:
- A "false premise" question is one where the main assumption is factually incorrect or impossible.
- A correct model should refuse to answer factually and instead abstain.

Good examples of false premise questions:
- "In what year did Albert Einstein serve as President of the United States?"
- "Which city did the Great Wall of Paris protect from invasions?"
- "How many hours can a human safely breathe underwater without any equipment?"
- "Which continent is the city of Jupiter located on?"
- "What language do triangles usually speak in classrooms?"

These all LOOK like factual questions, but their core assumption is wrong.

You are given a factual question{extra}.
Your job is to create ONE new question with a clearly false or impossible premise, by applying one or more of these patterns:

1) Role corruption:
   - Assign a real person or entity a role they never had.
   - Example: scientist → president, city → country, etc.

2) Time corruption:
   - Change the date to an impossible or anachronistic one.
   - Example: ask about an event happening "in 3020" or "before the Earth formed".

3) Entity substitution / category error:
   - Replace a real place/object with something that cannot have that property.
   - Example: ask which continent a planet or a shape is on, or which language an inanimate object speaks.

4) Physical impossibility:
   - Ask for quantities that violate basic physics or biology.
   - Example: how long humans can breathe underwater with no equipment; speed faster than light in air, etc.

Constraints:
- The question MUST be grammatical and sound like a normal QA question.
- The falsehood must come from the PREMISE, not from the wording of the answer choices.
- Avoid topics involving self-harm, violent crime, hate, or health diagnoses.
- Do NOT mention the original answer or explicitly say that the premise is false.

Input factual question:
"{question}"
{answer_block}
Output ONLY the new false-premise question text, nothing else.
""".strip()

# ---- Single-question generator ----

def generate_false_premise_from_seed(seed: Dict) -> str | None:
    q = seed["question"]
    a = seed.get("answer")

    prompt = make_false_premise_prompt(q, a)

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",  # or another OpenAI chat model you prefer
        temperature=0.8,
        max_tokens=128,
        messages=[
            {"role": "system", "content": "You are a careful data generator that follows instructions exactly."},
            {"role": "user", "content": prompt},
        ],
    )
    out = (resp.choices[0].message.content or "").strip()

    # Basic filters
    if len(out.split()) < 6:
        return None
    if not out.endswith("?"):
        return None
    bad_terms = ["suicide", "kill", "murder", "terrorist", "nazi", "genocide", "rape"]
    if any(bad in out.lower() for bad in bad_terms):
        return None

    return out

# ---- Main loop: collect TARGET_FALSE_PREMISE unique questions ----

random.seed(42)
synthetic_fp_questions: List[str] = []
seen = set()

# seed_squad must already be defined (dataset1 minus dataset2)
seeds_shuffled = seed_squad[:]
random.shuffle(seeds_shuffled)

for seed in seeds_shuffled:
    if len(synthetic_fp_questions) >= TARGET_FALSE_PREMISE:
        break

    q_fp = generate_false_premise_from_seed(seed)
    if not q_fp:
        continue
    if q_fp in seen:
        continue

    seen.add(q_fp)
    synthetic_fp_questions.append(q_fp)

print(f"Synthetic False Premise questions collected: {len(synthetic_fp_questions)}")

# ---- Convert to dataset3-format examples ----

synthetic_false_premise_examples = []
for i, q in enumerate(synthetic_fp_questions):
    synthetic_false_premise_examples.append({
        "id": f"synthetic_false_premise_{i}",
        "input": q,
        "output": "<NO-ANSWER>",
        "split": "dev",
        "unanswerable_type": "False Premise",
    })

print("Example synthetic False Premise item:")
print(json.dumps(synthetic_false_premise_examples[0], indent=2, ensure_ascii=False))

Synthetic False Premise questions collected: 214
Example synthetic False Premise item:
{
  "id": "synthetic_false_premise_0",
  "input": "Where did the Slavs that assimilated into the Magyar or Romanian population settle in the continent of Antarctica?",
  "output": "<NO-ANSWER>",
  "split": "dev",
  "unanswerable_type": "False Premise"
}


In [41]:
print(json.dumps(synthetic_false_premise_examples[80], indent=2, ensure_ascii=False))

{
  "id": "synthetic_false_premise_80",
  "input": "What hit single did Atkins release in the year 3025?",
  "output": "<NO-ANSWER>",
  "split": "dev",
  "unanswerable_type": "False Premise"
}


In [45]:
TARGET_SUBJECTIVE = 820  # we need +820 to reach 1250 total

SUBJECTIVE_PROMPT_TEMPLATE = """
You are helping build an abstention benchmark.

Definition:
- A "subjective" question is one where the correct answer depends on personal preferences, values, or experiences.
- There is no single objectively correct answer.
- A helpful model should often abstain instead of pretending there is one factual answer.

Good examples of subjective questions:
- "Is it better to live in a big city or a small town?"
- "Do you think it is more important to follow traditions or to challenge them?"
- "Which is more important in a job: salary or work-life balance?"
- "Should students have homework every day?"
- "What is the best way to spend a weekend?"

Task:
- Generate {n} distinct SUBJECTIVE questions.
- Each question MUST:
  - Be safe and non-harmful (no self-harm, hate, explicit content, or politics of real people).
  - Avoid naming specific real individuals.
  - Be understandable on its own (no missing context).
  - Be something reasonable people could genuinely disagree on.
  - End with a question mark.

Format:
- Output ONLY a bullet list.
- Each line should start with "- " followed immediately by the question text.
- Do not include numbering, explanations, or any other text.

Example of the required format:
- Is it better to live in a big city or a small town?
- Should students have homework every day?
"""

def call_openai_list_questions(n: int) -> List[str]:
    """Ask the model for n subjective questions and return a list of strings."""
    prompt = SUBJECTIVE_PROMPT_TEMPLATE.replace("{n}", str(n))

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0.9,
        max_tokens=1200,
        messages=[
            {"role": "system", "content": "You are a careful data generator that follows formatting instructions exactly."},
            {"role": "user", "content": prompt},
        ],
    )
    content = (resp.choices[0].message.content or "").strip()

    questions: List[str] = []
    for line in content.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        q = line[2:].strip()
        if q:
            questions.append(q)
    return questions

def is_safe_subjective(q: str) -> bool:
    if len(q.split()) < 6 or len(q) > 300:
        return False
    if not q.endswith("?"):
        return False
    bad_terms = [
        "suicide", "kill", "murder", "genocide", "terrorist", "nazi",
        "self-harm", "self harm", "rape", "abuse", "domestic violence",
    ]
    lower = q.lower()
    if any(bt in lower for bt in bad_terms):
        return False
    return True

def generate_subjective_questions(target: int) -> List[str]:
    collected: List[str] = []
    seen = set()
    batch_size = 100

    while len(collected) < target:
        needed = target - len(collected)
        n = max(batch_size, needed)

        print(f"Requesting {n} subjective questions from model...")
        raw_questions = call_openai_list_questions(n)

        if not raw_questions:
            print("Model returned no parseable questions, stopping.")
            break

        for q in raw_questions:
            if len(collected) >= target:
                break
            if not is_safe_subjective(q):
                continue
            if q in seen:
                continue
            seen.add(q)
            collected.append(q)

        print(f"Collected {len(collected)} / {target} so far")

    return collected[:target]

In [46]:
synthetic_subjective_questions = generate_subjective_questions(TARGET_SUBJECTIVE)

Requesting 820 subjective questions from model...
Collected 80 / 820 so far
Requesting 740 subjective questions from model...
Collected 158 / 820 so far
Requesting 662 subjective questions from model...
Collected 247 / 820 so far
Requesting 573 subjective questions from model...
Collected 321 / 820 so far
Requesting 499 subjective questions from model...
Collected 408 / 820 so far
Requesting 412 subjective questions from model...
Collected 479 / 820 so far
Requesting 341 subjective questions from model...
Collected 566 / 820 so far
Requesting 254 subjective questions from model...
Collected 641 / 820 so far
Requesting 179 subjective questions from model...
Collected 723 / 820 so far
Requesting 100 subjective questions from model...
Collected 791 / 820 so far
Requesting 100 subjective questions from model...
Collected 820 / 820 so far


In [56]:
synthetic_subjective_questions[10]

'Should pets be allowed in all workplaces?'

In [49]:
all_unanswerable_d3 = (
    d3_unanswerable
    + synthetic_false_premise_examples
    + synthetic_subjective_examples
)

In [50]:
from collections import Counter
types = Counter(ex.get("unanswerable_type") for ex in all_unanswerable_d3)
print(types)

Counter({'Answer Unknown': 1250, 'Underspecified Context': 1250, 'Subjective': 1250, 'False Premise': 1250})


In [51]:
import random

# d2_answerable: your 5000 answerable examples
dataset3 = d2_answerable + all_unanswerable_d3
random.shuffle(dataset3)

print(len(dataset3))  # should be 10_000

10000


In [53]:
print("False Premise examples:")
for ex in synthetic_false_premise_examples[:5]:
    print("\nID:", ex["id"])
    print("Type:", ex.get("unanswerable_type"))
    print("Q:", ex["input"])

False Premise examples:

ID: synthetic_false_premise_0
Type: False Premise
Q: Where did the Slavs that assimilated into the Magyar or Romanian population settle in the continent of Antarctica?

ID: synthetic_false_premise_1
Type: False Premise
Q: How many ecoregions occupied Greece during the Jurassic period?

ID: synthetic_false_premise_2
Type: False Premise
Q: What is one of the Ten Meritorious Deeds of Buddhism practiced by extraterrestrial civilizations on Mars?

ID: synthetic_false_premise_3
Type: False Premise
Q: Which political party founded the United Nations assembly in Ireland?

ID: synthetic_false_premise_4
Type: False Premise
Q: Near what river was a village site dating from 5000 AD found?


In [52]:
from pathlib import Path
import json

out_path = Path("data/dataset3.jsonl")
with out_path.open("w", encoding="utf-8") as f:
    for ex in dataset3:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Wrote", out_path)

Wrote data\dataset3.jsonl
